In [1]:
import glob
import numpy as np
import pandas as pd
import re

In [2]:
%load_ext autoreload

In [3]:
#Run this to reload the python file
%autoreload 2
from utils import *

In [7]:
# path file
#paths = glob.glob('./Data/GPM_raw/*v07*')
paths = glob.glob('./Data/GPM_raw/short_term/*')

### Extract metadata

In [9]:
start_date = '2017-10-04 00:00:00'
end_date = '2017-10-06 23:30:00'
frequency = '30min'

In [16]:
for path in paths:
    try:
        # read file
        df = pd.read_csv(path, header=None)
        print(f'reading: {path}')

        # extract the precipitation information
        df = df[8:]

        # assign column names
        df.columns = ['date', 'pcp']

        # find missing values
        missing_value = -9999.9
        df = missing_values(df, 'pcp', missing_value)

        # convert 'date' column to datetime
        df.loc[:, 'date'] = pd.to_datetime(df['date'])

        # set the 'date' column as the index
        df.set_index('date', inplace=True)

        # convert 'pcp' column to float
        df['pcp'] = df['pcp'].astype(float)

        # resample the time series to 1-hour frequency and aggregate using the sum
        df = df.resample('1H').sum()

        # reset the index and have 'date' as a column again
        df.reset_index(inplace=True)

        # save the dataframe as csv
        df.to_csv(f"./Data/harmonized/{path[26:-11]}_gpm_ST.csv")
        #df.to_csv(f"./Data/harmonized/{path[15:-7]}_ST.csv")
        #df.to_csv(f"./Data/harmonized/{path[15:-4]}.csv")

    except Exception as e:
        print(f"Error processing {path}: {str(e)}")
        continue

reading: ./Data/GPM_raw/short_term/69681_gpm_st.csv
reading: ./Data/GPM_raw/short_term/72163_gpm_st.csv
reading: ./Data/GPM_raw/short_term/76063_gpm_st.csv
reading: ./Data/GPM_raw/short_term/69713_gpm_st.csv
reading: ./Data/GPM_raw/short_term/72153_gpm_st.csv
reading: ./Data/GPM_raw/short_term/69679_gpm_st.csv
reading: ./Data/GPM_raw/short_term/69699_gpm_st.csv
reading: ./Data/GPM_raw/short_term/69633_gpm_st.csv
reading: ./Data/GPM_raw/short_term/69701_gpm_st.csv
reading: ./Data/GPM_raw/short_term/76055_gpm_st.csv
reading: ./Data/GPM_raw/short_term/69677_gpm_st.csv
reading: ./Data/GPM_raw/short_term/74053_gpm_st.csv
reading: ./Data/GPM_raw/short_term/72189_gpm_st.csv
reading: ./Data/GPM_raw/short_term/72149_gpm_st.csv
reading: ./Data/GPM_raw/short_term/69739_gpm_st.csv
reading: ./Data/GPM_raw/short_term/74051_gpm_st.csv
reading: ./Data/GPM_raw/short_term/74063_gpm_st.csv
reading: ./Data/GPM_raw/short_term/76065_gpm_st.csv
reading: ./Data/GPM_raw/short_term/69647_gpm_st.csv
reading: ./D

### Join all files into one

In [30]:
# path file
paths = glob.glob('./Data/harmonized/gpm_ST/*gpm*ST*csv')

In [31]:
df_comb = pd.DataFrame()

In [32]:
df_comb['date'] = pd.date_range(start=start_date, end=end_date, freq='H')

In [35]:
for path in paths:
    df = pd.read_csv(path)
    
    df_comb[path[25:-11]] = df['pcp']

In [36]:
path[25:-11]

'72183'

In [37]:
df_comb.to_csv('./Data/harmonized/unif_GPM_ST.csv', index=False)